### We run the optical from on 24CPU, it took around 8 minutes. You can generate it yourselves or use the already extracted values available in ./resources/

[info]
-- Apparently, there is a bug in some version of the OpenCV, and the inf magnitude is estimated ... Works with 4.10.82

In [1]:
from PIL import Image
import json
import os
import os.path as osp
import cv2
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

from joblib import Parallel, delayed

In [2]:
def process(current_frame: np.ndarray, previous_frame: np.ndarray, motion_threshold: float = None) -> dict:
    """
    Computes the degree of motion between the current and previous frames.

    Args:
        current_frame (np.ndarray): Current frame in grayscale.
        previous_frame (np.ndarray): Previous frame in grayscale.
        motion_threshold (float): Threshold to filter out low-motion noise.

    Returns:
        float: A single value representing the degree of motion in the frame.
    """
    if previous_frame is None:
        return {"score": None}

    # Calculate optical flow
    flow = cv2.calcOpticalFlowFarneback(
        prev=previous_frame,
        next=current_frame,
        flow=None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0,
    )

    # Compute magnitude and angle of flow vectors
    magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)
    magnitude = np.clip(magnitude, 0, 1e19)

    # Filter out small magnitudes (noise) using a threshold
    if motion_threshold is not None:
        significant_motion = magnitude[magnitude > motion_threshold]
    else:
        significant_motion = magnitude

    # Calculate the mean magnitude of significant motion
    motion_score = significant_motion.mean() if significant_motion.size > 0 else 0.0

    return {"score": motion_score}


In [ ]:
DATASET_DIR = "/media/marek/disk/datasets/sifco-accident"
METADATA_PATH = "/media/marek/disk/datasets/sifco-accident/labels.csv"
ANNOTATIONS_DIR = "./inference-yolo11x"

RESULTS_FOLDER = "./optical-flow"  # Folder to store results
results_file = os.path.join(RESULTS_FOLDER, "optical_flow.pkl")

TARGET_FPS = 5

os.makedirs(RESULTS_FOLDER, exist_ok=True)


In [4]:
annotations = pd.read_csv(METADATA_PATH)

detection_paths, video_paths = [], []
for i, row in annotations.iterrows():
    path = row["path"]
    detections_path = os.path.join(ANNOTATIONS_DIR, osp.dirname(path), Path(path).stem + ".json")
    assert osp.exists(detections_path), detections_path
    detection_paths.append(detections_path)

    video_path = osp.join(DATASET_DIR, path)
    assert osp.exists(video_path), video_path
    video_paths.append(video_path)


annotations["detections_path"] = detection_paths
annotations["video_path"] = video_paths


In [15]:
def process_frame_sequence(video_path: str) -> dict:
    """
    Process a single video file to compute optical flow at the target FPS.

    Returns:
        dict: Dictionary containing video filename as the key and optical flow results as the value.
    """
    if not os.path.exists(video_path):
        print(f"Video {video_path} does not exist.")
        return {osp.basename(video_path): None}  # Skip processing if the file doesn't exist

    cap = cv2.VideoCapture(video_path)
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_skip = min(int(original_fps / TARGET_FPS), 1)

    previous_frame = None
    video_results = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_skip == 0:
            print("bad frame skip")
            break

        if frame_count % frame_skip == 0:
            current_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            result = process(
                current_frame, previous_frame
            )  # Assuming process() is defined elsewhere
            result["frame"] = frame_count
            video_results.append(result)
            previous_frame = current_frame

        frame_count += 1

    cap.release()
    return {osp.basename(video_path): video_results}


In [24]:
# missing = []
#
# mask = annotations["path"].apply(lambda x: osp.basename(x) in missing)
# annotations = annotations[mask]

results_list = Parallel(n_jobs=-1)(
    delayed(process_frame_sequence)(filename) for filename in tqdm(annotations["video_path"], total=len(annotations["video_path"]))
)

# Combine results from all videos
results = {}
for result in results_list:
    results.update(result)

torch.save(results, results_file)


100%|██████████| 2/2 [00:00<00:00, 3148.88it/s]


## Merge optical flows of processed videos

In [25]:
optical_flow_files = [
    "optical_flow_0-800.pkl",
    "optical_flow_800-1200.pkl",
    "optical_flow_1200-1500.pkl",
    "optical_flow_1500-end.pkl",
    "optical_flow_fix.pkl"
]

data = {}
for file in optical_flow_files:
    optical_flow_path = osp.join(RESULTS_FOLDER, file)
    optical_flow_data = torch.load(optical_flow_path, weights_only=False)
    data.update(optical_flow_data)

print(len(data))
torch.save(data, osp.join(RESULTS_FOLDER, "optical_flow.pkl"))


1770


In [22]:
# Load optical flow data
optical_flow_data = torch.load(
    osp.join(RESULTS_FOLDER, "optical_flow.pkl"), weights_only=False
)
records = []

# Iterate over the optical flow data to prepare a list of records
for video_filename, frames in optical_flow_data.items():
    # Correct video name format if needed

    for frame_info in frames:
        records.append(
            {
                "filename": video_filename,
                "frame": frame_info["frame"],
                "score": frame_info["score"],
            }
        )

# Create a DataFrame from the records
optical_flow_df = pd.DataFrame(records)

# Extract the video name from the filename
optical_flow_df["video"] = optical_flow_df["filename"].apply(
    lambda filename: filename.split(".")[0]
)

optical_flow_df

,filename,frame,score,video
0,(FATAL) Georgia： Distracted Driver Crashes Int...,0,NaN,(FATAL) Georgia： Distracted Driver Crashes Int...
1,(FATAL) Georgia： Distracted Driver Crashes Int...,6,0.506993,(FATAL) Georgia： Distracted Driver Crashes Int...
2,(FATAL) Georgia： Distracted Driver Crashes Int...,12,0.490422,(FATAL) Georgia： Distracted Driver Crashes Int...
3,(FATAL) Georgia： Distracted Driver Crashes Int...,18,0.524868,(FATAL) Georgia： Distracted Driver Crashes Int...
4,(FATAL) Georgia： Distracted Driver Crashes Int...,24,0.510733,(FATAL) Georgia： Distracted Driver Crashes Int...
...,...,...,...,...
519114,"＂Special＂ Driver DESPERATE To Reach Exit, Cras...",445,0.104723,"＂Special＂ Driver DESPERATE To Reach Exit, Cras..."
519115,"＂Special＂ Driver DESPERATE To Reach Exit, Cras...",446,0.111968,"＂Special＂ Driver DESPERATE To Reach Exit, Cras..."
519116,"＂Special＂ Driver DESPERATE To Reach Exit, Cras...",447,0.116988,"＂Special＂ Driver DESPERATE To Reach Exit, Cras..."
519117,"＂Special＂ Driver DESPERATE To Reach Exit, Cras...",448,0.147047,"＂Special＂ Driver DESPERATE To Reach Exit, Cras..."
